# Vague 4 — Calibration directe des poids v2 sur Propluvia

**Objectif :** remplacer les poids AHP "expert" (`w_bws=0.50, w_a=0.36, w_gws=0.14`) par les poids **optimaux empiriquement** — ceux qui maximisent ρ_Spearman(H_v2, niveau Propluvia) sur les 453 zones d'alerte.

**Méthode :**
1. Train/test split **stratifié** (70 % entraînement, 30 % validation, jamais touché pendant l'optim).
2. Optimisation par `scipy.optimize.differential_evolution` (Spearman ρ non différentiable → algo sans dérivée).
3. Variables optimisées : `w_bws`, `w_a` (et `w_gws = 1 - w_bws - w_a` par contrainte de somme).
4. `β_0`, `τ_Boltz`, `C_asym`, `EPSILON` figés (on optimise *que* les poids ; on étendra ensuite si besoin).

**Garde-fou :** le ρ qui compte est celui du test set. Si train >> test, c'est de l'overfit.

---

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from scipy.optimize import differential_evolution
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
CSV = ROOT / "data" / "wave3" / "zones_h_local_2022.csv"
df = pd.read_csv(CSV, dtype={"code_dept": str, "zone_id": str})
df = df.dropna(subset=["x_bws", "x_gws", "x_a", "spei3", "spei12", "niveau_int"]).copy()
print(f"Zones valides : {len(df)}")
print(df[["x_bws", "x_a", "x_gws", "spei3", "spei12", "niveau_int"]].describe().round(3))

## 1. Train / test split

Split stratifié sur `niveau_int` pour garder la même distribution de gravité dans les deux sets.

In [ ]:
df_train, df_test = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["niveau_int"]
)
print(f"Train : {len(df_train)}  |  Test : {len(df_test)}")
print()
print("Distribution Propluvia par split :")
print(pd.concat([
    df_train["niveau_int"].value_counts().rename("train"),
    df_test["niveau_int"].value_counts().rename("test"),
], axis=1).sort_index())

## 2. Modèle v2 paramétrable

Les formules sont identiques à HlocalV3 — seule différence : les poids sont injectables.

In [ ]:
TAU_BOLTZ = 5.0
C_ASYM = 2.0
BETA_0 = 0.70

def f_asym(spei, c=C_ASYM):
    s = np.asarray(spei, dtype=float)
    out = np.where(s >= 0, 0.0, -s / (c + np.abs(s)))
    return np.clip(out, 0, 1)

def s_struct_v2_arr(x_bws, x_a, x_gws, w_bws, w_a, w_gws):
    return (w_bws * x_bws + w_a * x_a + w_gws * x_gws) / (w_bws + w_a + w_gws)

def s_conj_boltz(d3, d12, tau=TAU_BOLTZ):
    e3, e12 = np.exp(tau * d3), np.exp(tau * d12)
    return (d3 * e3 + d12 * e12) / (e3 + e12)

def H_v2_compute(df_, w_bws, w_a, w_gws, beta_0=BETA_0):
    x_bws = df_["x_bws"].to_numpy()
    x_a   = df_["x_a"].to_numpy()
    x_gws = df_["x_gws"].to_numpy()
    spei3 = df_["spei3"].to_numpy()
    spei12 = df_["spei12"].to_numpy()
    s_struct = s_struct_v2_arr(x_bws, x_a, x_gws, w_bws, w_a, w_gws)
    d3, d12 = f_asym(spei3), f_asym(spei12)
    s_conj = s_conj_boltz(d3, d12)
    beta_t = beta_0 * (1 - s_conj)
    return np.clip(beta_t * s_struct + (1 - beta_t) * s_conj, 0, 1)

## 3. Baseline — poids AHP "expert"

In [ ]:
W_AHP = (0.50, 0.36, 0.14)   # bws, a, gws  (verbatim HlocalV3)

H_train_ahp = H_v2_compute(df_train, *W_AHP)
H_test_ahp  = H_v2_compute(df_test,  *W_AHP)
rho_train_ahp, p_train_ahp = stats.spearmanr(H_train_ahp, df_train["niveau_int"])
rho_test_ahp,  p_test_ahp  = stats.spearmanr(H_test_ahp,  df_test["niveau_int"])

print(f"Poids AHP : w_bws={W_AHP[0]}, w_a={W_AHP[1]}, w_gws={W_AHP[2]}")
print(f"  rho train = {rho_train_ahp:+.3f}  (p={p_train_ahp:.2e})")
print(f"  rho test  = {rho_test_ahp:+.3f}  (p={p_test_ahp:.2e})")

## 4. Optimisation

On cherche `(w_bws, w_a)` ∈ [0, 1]², avec `w_gws = 1 - w_bws - w_a ≥ 0`. Si la contrainte n'est pas respectée, on pénalise.

On maximise ρ sur **train uniquement**.

In [ ]:
P_train = df_train["niveau_int"].to_numpy()

def neg_spearman_train(params):
    w_bws, w_a = params
    w_gws = 1.0 - w_bws - w_a
    if w_gws < 0 or w_bws < 0 or w_a < 0:
        return 1.0   # pénalité
    H = H_v2_compute(df_train, w_bws, w_a, w_gws)
    rho, _ = stats.spearmanr(H, P_train)
    return -rho if not np.isnan(rho) else 1.0

result = differential_evolution(
    neg_spearman_train,
    bounds=[(0.01, 0.99), (0.01, 0.99)],
    seed=42, maxiter=200, tol=1e-5, polish=True,
)
w_bws_opt, w_a_opt = result.x
w_gws_opt = 1.0 - w_bws_opt - w_a_opt
rho_train_opt = -result.fun

print(f"Convergence : {result.success}")
print(f"Poids optimaux : w_bws={w_bws_opt:.3f}, w_a={w_a_opt:.3f}, w_gws={w_gws_opt:.3f}")
print(f"  rho train = {rho_train_opt:+.3f}")

## 5. Évaluation **honnête** sur le test set

C'est ce ρ qui compte. S'il est >> ρ AHP, on a vraiment gagné. S'il est proche, on a fitté le bruit.

In [ ]:
H_test_opt = H_v2_compute(df_test, w_bws_opt, w_a_opt, w_gws_opt)
rho_test_opt, p_test_opt = stats.spearmanr(H_test_opt, df_test["niveau_int"])

print("Comparaison sur le set TEST (jamais utilisé pendant l'optim) :")
print(f"  AHP expert  : rho_test = {rho_test_ahp:+.3f}")
print(f"  Optimisé    : rho_test = {rho_test_opt:+.3f}  (p={p_test_opt:.2e})")
gain = rho_test_opt - rho_test_ahp
print(f"  Gain        : {gain:+.3f}")
if gain > 0.02:
    print("  → Vrai gain.")
elif abs(gain) < 0.02:
    print("  → Pas de difference significative — les poids AHP sont OK.")
else:
    print("  → L'optim a degrade le test → overfit.")

## 6. Visualisation des deux modèles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, H, lbl, rho in [
    (axes[0], H_test_ahp, f"AHP (rho_test={rho_test_ahp:+.3f})", rho_test_ahp),
    (axes[1], H_test_opt, f"Optim (rho_test={rho_test_opt:+.3f})", rho_test_opt),
]:
    ax.scatter(H, df_test["niveau_int"] + np.random.uniform(-0.15, 0.15, len(H)),
               alpha=0.5, s=18, color="#0277BD")
    ax.set_xlabel("H_local v2")
    ax.set_ylabel("Niveau Propluvia (jitter)")
    ax.set_title(lbl)
    ax.grid(alpha=0.3)
fig.suptitle("H_local v2 vs niveau Propluvia (test set)")
plt.tight_layout()
plt.show()

## 7. Bootstrap IC95 % du gain

Le gain ρ_optim − ρ_AHP est-il statistiquement réel ou dans le bruit du split ?

In [ ]:
rng = np.random.default_rng(42)
n_boot = 5000
n = len(df_test)
deltas = []
for _ in range(n_boot):
    idx = rng.integers(0, n, size=n)
    sub = df_test.iloc[idx]
    H_a = H_v2_compute(sub, *W_AHP)
    H_o = H_v2_compute(sub, w_bws_opt, w_a_opt, w_gws_opt)
    r_a, _ = stats.spearmanr(H_a, sub["niveau_int"])
    r_o, _ = stats.spearmanr(H_o, sub["niveau_int"])
    if not np.isnan(r_a) and not np.isnan(r_o):
        deltas.append(r_o - r_a)
deltas = np.array(deltas)
lo, hi = np.percentile(deltas, [2.5, 97.5])
med = np.median(deltas)
print(f"Delta_rho (optim - AHP) sur test, mediane = {med:+.3f}")
print(f"IC95 = [{lo:+.3f}, {hi:+.3f}]")
print(f"Zero dans IC95 : {'OUI (gain non significatif)' if lo <= 0 <= hi else 'NON (gain reel)'}")

## 8. Sanity check — sensibilité du seed

Re-fait l'optim avec 5 splits différents pour voir si le résultat est stable.

In [ ]:
results = []
for seed in range(5):
    df_tr, df_te = train_test_split(df, test_size=0.30, random_state=seed,
                                    stratify=df["niveau_int"])
    P_tr = df_tr["niveau_int"].to_numpy()
    def obj(p, df_=df_tr, P_=P_tr):
        wb, wa = p
        wg = 1 - wb - wa
        if wg < 0 or wb < 0 or wa < 0: return 1.0
        H = H_v2_compute(df_, wb, wa, wg)
        r, _ = stats.spearmanr(H, P_)
        return -r if not np.isnan(r) else 1.0
    res = differential_evolution(obj, bounds=[(0.01, 0.99), (0.01, 0.99)],
                                 seed=seed, maxiter=100, tol=1e-4, polish=True)
    wb, wa = res.x; wg = 1 - wb - wa
    H_te = H_v2_compute(df_te, wb, wa, wg)
    r_te, _ = stats.spearmanr(H_te, df_te["niveau_int"])
    H_te_ahp = H_v2_compute(df_te, *W_AHP)
    r_te_ahp, _ = stats.spearmanr(H_te_ahp, df_te["niveau_int"])
    results.append({"seed": seed, "w_bws": wb, "w_a": wa, "w_gws": wg,
                    "rho_test_optim": r_te, "rho_test_ahp": r_te_ahp,
                    "gain": r_te - r_te_ahp})
rdf = pd.DataFrame(results)
print(rdf.round(3).to_string(index=False))
print()
print(f"Gain median = {rdf['gain'].median():+.3f}")
print(f"Gain min/max = [{rdf['gain'].min():+.3f}, {rdf['gain'].max():+.3f}]")

## 9. Lecture des résultats

**Trois cas possibles :**

1. **Gain >> 0 et stable entre seeds** → les poids AHP étaient sous-optimaux. Adopter les nouveaux poids.
2. **Gain ≈ 0 sur le test** → les poids AHP capturaient déjà l'essentiel. Pour gagner plus, il faut ajouter des inputs (Hub'Eau piézométrie, Banque Hydro débits) ou changer de modèle (Random Forest).
3. **Gain énorme sur train mais faible/négatif sur test** → overfit. Le modèle v2 est trop simple pour exploiter plus de signal qu'il en a déjà.

Dans tous les cas, ce notebook **fixe une borne empirique** : le ρ maximal atteignable avec H_v2 et les inputs actuels.

Si tu veux dépasser ce plafond → **vague 5 = Random Forest sur les mêmes inputs** pour mesurer le potentiel sans contrainte structurelle.